In [ ]:
from pathlib import Path
import urllib.request

In [ ]:
def download_shakespeare_text():
    path = Path("datasets/shakespeare/shakespeare.txt")
    if not path.is_file():
        path.parent.mkdir(parents=True, exist_ok=True)
        url ="https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
        urllib.request.urlretrieve(url, path)
    return path.read_text()

shakespeare_text = download_shakespeare_text()


In [ ]:
print(shakespeare_text[:80])

In [ ]:
vocab = sorted(set(shakespeare_text.lower()))
"".join(vocab)

In [ ]:
char_to_id = {char: index for index, char in enumerate(vocab)}
id_to_char = {index: char for index, char in enumerate(vocab)}

In [ ]:
import torch

def encode_text(text):
    return torch.tensor([char_to_id[char] for char in text.lower()])

def decode_text(char_ids):
    return "".join([id_to_char[char_id.item()] for char_id in char_ids])

In [ ]:
encoded = encode_text("Hello, world!")
encoded

In [ ]:
decode_text(encoded)

In [ ]:
from torch.utils.data import Dataset, DataLoader
class CharDataset(Dataset):

    def __init__(self, text, window_length):
        self.encoded_text = encode_text(text)
        self.window_length = window_length

    def __len__(self):
        return len(self.encoded_text) - self.window_length
    
    def __getitem__(self, idx):
        if idx >= len(self):
            raise IndexError("dataset index out of range")
        
        end = idx + self.window_length
        window = self.encoded_text[idx:end]
        target = self.encoded_text[idx+1 : end+1]
        return window, target

In [ ]:
window_length = 50 # reduce to faster training
batch_size = 256 # reduce if your GPU cannot handle such a large batch size

train_set = CharDataset(shakespeare_text[:1_000_000], window_length)
valid_set = CharDataset(shakespeare_text[1_000000:1_060_000], window_length)
test_set = CharDataset(shakespeare_text[1_060_000:], window_length)

train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True)
valid_loader = DataLoader(valid_set, batch_size=batch_size)
test_loader = DataLoader(test_set, batch_size=batch_size)

Model and Embedding

In [ ]:
import torch.nn as nn
torch.manual_seed(42)

In [ ]:
class ShakespeareModel(nn.Module):
    def __init__(self, vocab_size, n_layers=2, embed_dim=10, hidden_dim=128,
                 dropout=0.1):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim)
        self.gru = nn.GRU(embed_dim, hidden_dim, num_layers=n_layers,
                          batch_first=True, dropout=dropout)
        self.output = nn.Linear(hidden_dim, vocab_size)

    def forward(self, X):
        embeddings = self.embed(X)
        outputs, _states = self.gru(embeddings)
        return self.output(outputs).permute(0, 2, 1)
    

In [ ]:
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(device)

In [ ]:
model = ShakespeareModel(len(vocab)).to(device)

In [ ]:
import torch.optim as optim

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

model.train()
epochs = 20               # Adjust based on your computational ability.
best_val_loss = float('inf')
patience = 5              # stop if no improvement for 5 epochs
no_improve = 0

for epoch in range(1, epochs + 1):
    # Training
    model.train()
    train_loss = 0
    for batch_X, batch_y in train_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        optimizer.zero_grad()
        outputs = model(batch_X)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
    avg_train_loss = train_loss / len(train_loader)

    # Validation
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for val_X, val_y in valid_loader:
            val_X, val_y = val_X.to(device), val_y.to(device)
            val_outputs = model(val_X)
            val_loss += criterion(val_outputs, val_y).item()
    avg_val_loss = val_loss / len(valid_loader)

    print(f"Epoch {epoch:2d} | train loss: {avg_train_loss:.4f} | val loss: {avg_val_loss:.4f}")

    # Early stopping
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        no_improve = 0
    else:
        no_improve += 1
        if no_improve >= patience:
            print(f"Early stopping at epoch {epoch}")
            break

print("Training finished!")

In [ ]:
model.eval() 
text = "To be or not to b"
encoded_text = encode_text(text).unsqueeze(dim=0).to(device)

with torch.no_grad():
    Y_logits = model(encoded_text)
    predicted_char_id = Y_logits[0, :,-1].argmax().item()
    predicted_char = id_to_char[predicted_char_id]

In [ ]:
predicted_char

In [ ]:
import torch.nn.functional as F

def next_char(model, text, temperature = 1):
    encoded_text = encode_text(text).unsqueeze(dim=0).to(device)
    with torch.no_grad():
        Y_logits = model(encoded_text)
        Y_probas = F.softmax(Y_logits[0, :, -1] / temperature, dim=-1)
        predicted_char_id = torch.multinomial(Y_probas, num_samples=1).item()

    return id_to_char[predicted_char_id]

In [ ]:
def extend_text(model, text, n_chars=80, temperature=1):
    for _ in range(n_chars):
        text += next_char(model, text, temperature)
    return text

In [ ]:
print(extend_text(model, "To be or not to b", temperature=0.4))